In [1]:
# librerias
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
import pandas as pd

#leer dataset
df = pd.read_csv("Dataset Arreglado.csv", sep=',', quotechar='"')


In [3]:
# probar
display(df.head())

,USER_KEY,TARGET_KEY,TYPE_KEY,TEXTO_RECLAMO
0,DIAZ FLORES KARINA,20100043140,RECLAMO,El celular que recibí no corresponde al modelo...
1,CASTILLO PALOMINO VICTOR,20517352573,DENUNCIA,Hago de su conocimiento que el suscrito ha sid...
2,MENDOZA RAMIREZ CARMEN,20537630222,QUEJA,A quien corresponda: me dirijo a usted respetu...
3,MORALES GUERRERO GIULIANA,20508565934,RECLAMO,Me están cobrando una prima mensual de S/350 q...
4,CASTRO ROJAS JESSICA,20339340540,DENUNCIA,Me reportaron a central de riesgos por una deu...


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 4 columns):
 #   Column         Non-Null Count    Dtype 
---  ------         --------------    ----- 
 0   USER_KEY       1000000 non-null  object
 1   TARGET_KEY     1000000 non-null  int64 
 2   TYPE_KEY       1000000 non-null  object
 3   TEXTO_RECLAMO  1000000 non-null  object
dtypes: int64(1), object(3)
memory usage: 30.5+ MB




---


# ANÁLISIS DE BOT - REPETICIÓN DE FRASES
Si la misma frase se repite más de N veces en el registro, se interpreta como un spam.

Siendo N = 2, se observan 40148 frases que se consideran spam.

*(N=2, significa que si una frase se repite 3 o más veces, se considera spam)*

In [5]:
# Contar la frecuencia de cada texto en la columna 'TEXTO_RECLAMO'
repeated_texts = df['TEXTO_RECLAMO'].value_counts()

# Filtrar para mostrar solo los textos que se repiten 3 o más veces
duplicate_texts_df = repeated_texts[repeated_texts >= 3].reset_index()
duplicate_texts_df.columns = ['TEXTO_RECLAMO', 'count']

# Mostrar los textos duplicados y sus recuentos
print(f"Se encontraron {len(duplicate_texts_df)} textos que se repiten 3 o más veces.")
display(duplicate_texts_df.head(10)) # Mostrar los primeros 10

Se encontraron 40148 textos que se repiten 3 o más veces.


,TEXTO_RECLAMO,count
0,El suscrito se dirige a usted por medio de la ...,17843
1,A quien corresponda: me dirijo a usted respetu...,17619
2,Mediante la presente el suscrito hace de su co...,17547
3,Me dirijo a usted respetuosamente para solicit...,17419
4,Por medio de la presente hago de su conocimien...,17215
5,Me dirijo a usted por medio de la presente par...,13365
6,Hago de su conocimiento que el suscrito ha sid...,13277
7,Por medio de la presente y de conformidad con ...,13200
8,"Por medio de la presente, el suscrito pone en ...",13197
9,Respetuosamente me dirijo a usted por medio de...,13181


Dentro del conjunto de datos, solo existen 205952 frases que no se repiten ni una sola vez.

In [6]:
# Filtrar para mostrar solo los textos que no se repiten (aparecen exactamente una vez)
non_duplicate_texts_df = repeated_texts[repeated_texts == 1].reset_index()
non_duplicate_texts_df.columns = ['TEXTO_RECLAMO', 'count']

# Mostrar los textos únicos y sus recuentos
print(f"Se encontraron {len(non_duplicate_texts_df)} textos que aparecen exactamente una vez.")
display(non_duplicate_texts_df.head(10)) # Mostrar los primeros 10

Se encontraron 205952 textos que aparecen exactamente una vez.


,TEXTO_RECLAMO,count
0,Realicé un pago de S/500 hace 10 días y aún no...,1
1,Necesito la devolución de mi dinero por el art...,1
2,Realicé un pago de S/1500 hace 20 días y aún n...,1
3,Me cobraron S/500 más que el precio indicado e...,1
4,Realicé un pago de S/1200 hace 7 días y aún no...,1
5,La velocidad de mi internet es de 3 Mbps cuand...,1
6,Solicito la devolución de mi dinero por el pro...,1
7,Necesito la devolución de mi dinero por el art...,1
8,Solicito la devolución de mi dinero por el pro...,1
9,Requiero la devolución de mi dinero por el pro...,1




---


# ANÁLISIS DE BOT - BIGRAMAS CLAVE DE BOTS
Dentro de las frases que más se repiten, hay conjuntos de palabras que se utilizan de sobremedida.

Se busca descubrir esa lista de bigramas más repetidas entre los mensajes spam.

In [7]:
duplicate_texts_df.iloc[10000]

,10000
TEXTO_RECLAMO,Bloquearon mi cuenta de ahorros sin previo avi...
count,10


In [8]:
top_texts = repeated_texts.head(10000)  # Las 10000 frases mas repetidas

stopwords = {
    "de", "la", "el", "los", "las",
    "y", "o", "con", "sin", "pero",
    "ya", "se", "su", "más", "por",
    "para", "del", "que", "una",
    "uno", "unos", "unas", "al",
    "es", "en", "un"
} # palabras que no se deben considerar

frecuencias = {}  # diccionario de palabras

# contar repeticion de palabras
# Recorrer todos los textos
for text in top_texts.index:
  # Pasar a minúsculas
  text = text.lower()
  # Separar palabras
  words = text.split()
  # Limpiar palabras
  clean_words = []

  for word in words:
      # Eliminar puntuación básica
      word = word.strip(".,;:!?()[]{}\"'")
      # Ignorar números
      if word.isdigit():
          continue
      # Ignorar palabras cortas
      if len(word) <= 2:
          continue
      # Ignorar stopwords
      if word in stopwords:
          continue
      clean_words.append(word)

  # Crear bigramas
  for i in range(len(clean_words) - 1):
      bigrama = clean_words[i] + " " + clean_words[i+1]

      # Contar frecuencia
      if bigrama in frecuencias:
          frecuencias[bigrama] += 1
      else:
          frecuencias[bigrama] = 1

# Ordenar bigramas por frecuencia
sorted_bigrama = sorted(
    frecuencias.items(),
    key=lambda x: x[1],
    reverse=True
)

# Mostrar TOP 100
print("TOP 100 BIGRAMAS MÁS REPETIDOS:\n")

for bigrama, freq in sorted_bigrama[:100]:
    print(f"{bigrama} -> {freq}")

TOP 100 BIGRAMAS MÁS REPETIDOS:

desde hace -> 2065
llamé veces -> 1572
hace días -> 1292
hace semanas -> 1267
lleva días -> 1066
ese tipo -> 1063
están cobrando -> 1049
tickets soporte -> 998
soporte fueron -> 998
fueron cerrados -> 998
cerrados automáticamente -> 998
automáticamente nadie -> 998
nadie revisara -> 998
esperé horas -> 991
horas línea -> 991
línea atención -> 991
atención cliente -> 991
cliente cuando -> 991
cuando fin -> 991
fin atendieron -> 991
atendieron volvieron -> 991
volvieron poner -> 991
poner espera -> 991
incumplimiento servicio -> 868
solicito cancelación -> 867
cancelación contrato -> 867
contrato penalidad -> 867
penalidad incumplimiento -> 867
servicio parte -> 867
veces rastrear -> 855
rastrear paquete -> 855
paquete sistema -> 855
sistema dice -> 855
dice está -> 855
está tránsito -> 855
tránsito desde -> 855
exijo restablecimiento -> 847
restablecimiento inmediato -> 847
inmediato servicio -> 847
servicio devolución -> 847
devolución proporcional -> 8

# ANÁLISIS DE BIGRAMAS EN MENSAJES ÚNICOS
Calculamos los bigramas de los mensajes que no se repiten para comparar y filtrar falsos positivos de spam.

In [9]:
top_non_duplicate = non_duplicate_texts_df['TEXTO_RECLAMO']

frecuencias_no_spam = {}

for text in top_non_duplicate:
    text = str(text).lower()
    words = text.split()
    clean_words = []

    for word in words:
        word = word.strip(".,;:!?()[]{}\"' ")
        if word.isdigit() or len(word) <= 2 or word in stopwords:
            continue
        clean_words.append(word)

    for i in range(len(clean_words) - 1):
        bigrama = clean_words[i] + " " + clean_words[i+1]
        frecuencias_no_spam[bigrama] = frecuencias_no_spam.get(bigrama, 0) + 1

sorted_bigrama_no_spam = sorted(
    frecuencias_no_spam.items(),
    key=lambda x: x[1],
    reverse=True
)

print("TOP 20 BIGRAMAS EN MENSAJES ÚNICOS:")
for bigrama, freq in sorted_bigrama_no_spam[:20]:
    print(f"{bigrama} -> {freq}")

TOP 20 BIGRAMAS EN MENSAJES ÚNICOS:
devolución dinero -> 104793
funciona desde -> 86711
desde primer -> 86711
primer día -> 86711
dinero producto -> 82337
producto defectuoso -> 82337
día adjunto -> 61804
adjunto boleta -> 61804
boleta número -> 61804
solicito devolución -> 42824
defectuoso compré -> 41350
hace días -> 33084
devuelvan dinero -> 31181
dijeron debía -> 30116
requiero devolución -> 23106
exijo devolución -> 22769
devolución inmediata -> 22007
cobrados forma -> 22007
forma indebida -> 22007
compré tienda -> 20817


### Diferencia de Frecuencias de Bigramas
Restamos la frecuencia de bigramas encontrados en mensajes únicos a la frecuencia de los mensajes repetidos (spam) para encontrar patrones exclusivos de bots.

In [10]:
dif_frecuencias = dict(frecuencias)

for bigrama, freq in frecuencias_no_spam.items():
    if bigrama in dif_frecuencias:
        dif_frecuencias[bigrama] -= freq
    else:
        dif_frecuencias[bigrama] = -freq

sorted_dif_bigramas = sorted(
    dif_frecuencias.items(),
    key=lambda x: x[1],
    reverse=True
)

print("TOP 200 BIGRAMAS CON MAYOR DIFERENCIA (SPAM - ÚNICOS):")
for bigrama, diff in sorted_dif_bigramas[:200]:
    print(f"{bigrama} -> {diff}")

TOP 200 BIGRAMAS CON MAYOR DIFERENCIA (SPAM - ÚNICOS):
desde hace -> 1778
hace semanas -> 1059
veces rastrear -> 855
rastrear paquete -> 855
paquete sistema -> 855
sistema dice -> 855
dice está -> 855
está tránsito -> 855
tránsito desde -> 855
incumplimiento servicio -> 721
solicito cancelación -> 720
cancelación contrato -> 720
contrato penalidad -> 720
penalidad incumplimiento -> 720
servicio parte -> 720
ejecutivo cuenta -> 719
cuenta contesta -> 719
contesta responde -> 719
responde correos -> 719
correos desde -> 719
exijo restablecimiento -> 698
restablecimiento inmediato -> 698
inmediato servicio -> 698
servicio devolución -> 698
devolución proporcional -> 698
proporcional días -> 698
días conexión -> 698
pido verifiquen -> 687
verifiquen velocidad -> 687
velocidad real -> 687
real domicilio -> 687
domicilio ajusten -> 687
ajusten contratado -> 687
paquete lleva -> 657
días tránsito -> 657
tránsito cuando -> 657
cuando plazo -> 657
plazo acordado -> 657
acordado era -> 657
era d

In [11]:
# guardar los 200 bigramas mas frecuentes
top_200_bigrams = []

for bigrama, freq in sorted_dif_bigramas[:200]:
    top_200_bigrams.append(bigrama)

print(top_200_bigrams)

['desde hace', 'hace semanas', 'veces rastrear', 'rastrear paquete', 'paquete sistema', 'sistema dice', 'dice está', 'está tránsito', 'tránsito desde', 'incumplimiento servicio', 'solicito cancelación', 'cancelación contrato', 'contrato penalidad', 'penalidad incumplimiento', 'servicio parte', 'ejecutivo cuenta', 'cuenta contesta', 'contesta responde', 'responde correos', 'correos desde', 'exijo restablecimiento', 'restablecimiento inmediato', 'inmediato servicio', 'servicio devolución', 'devolución proporcional', 'proporcional días', 'días conexión', 'pido verifiquen', 'verifiquen velocidad', 'velocidad real', 'real domicilio', 'domicilio ajusten', 'ajusten contratado', 'paquete lleva', 'días tránsito', 'tránsito cuando', 'cuando plazo', 'plazo acordado', 'acordado era', 'era días', 'envié paquete', 'paquete contenido', 'contenido valorado', 'llegó evidentes', 'evidentes signos', 'signos haber', 'haber sido', 'sido abierto', 'abrí reclamo', 'solo dijeron', 'dijeron estaban', 'estaban 

Se utiliza la lista de los 200 bigramas más frecuentes entre mensajes de spam para encontrar coincidencias.

Si un texto posee M bigramas o más, se considera un mensaje de spam.

In [12]:
# Deteccion de spam en base a bigramas con preprocesamiento completo
def detect_spam(text, M):
    text = str(text).lower()
    words = text.split()
    clean_words = []

    for word in words:
        word = word.strip(".,;:!?()[]{}\"' ")
        if word.isdigit() or len(word) <= 2 or word in stopwords:
            continue
        clean_words.append(word)

    text_bigrams = []
    for i in range(len(clean_words) - 1):
        bigram = clean_words[i] + " " + clean_words[i+1]
        text_bigrams.append(bigram)

    coind = 0
    for bigram in top_200_bigrams:
        if bigram in text_bigrams:
            coind += 1

    # Regla: M o más bigramas sospechosos => spam
    return coind >= M

# CONTAR SPAM EN EL DATAFRAME
# Usamos M=2 como criterio
df['IS_SPAM'] = df['TEXTO_RECLAMO'].apply(lambda x: detect_spam(x, 4))
spam_count = df['IS_SPAM'].sum()

print(f"Cantidad de mensajes spam detectados: {spam_count}")
print(f"Porcentaje de spam: {(spam_count/len(df))*100:.2f}%")

Cantidad de mensajes spam detectados: 323609
Porcentaje de spam: 32.36%


### Exportar bigramas
Guardamos la lista de bigramas en un archivo JSON para que sea fácil de importar.

In [13]:
import json

with open('top_200_bigrams.json', 'w', encoding='utf-8') as f:
    json.dump(top_200_bigrams, f, ensure_ascii=False, indent=2)

print("Los bigramas fueron guardados en: 'top_200_bigrams.json'.")

Los bigramas fueron guardados en: 'top_200_bigrams.json'.
